# Colab runner for the experiment ladder

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Fill in `REPO_URL` below. If the repo is private, add a Colab secret named `GH_TOKEN`
   (key icon in the left bar; a fine-grained, read-only token for this repo) and allow notebook access.
3. Set `STAGE` to the stage you were asked to run. At the end a `results_<STAGE>.zip` downloads;
   drop it into the `Sinai/` folder on your Mac.

If the runtime disconnects, just **Run all** again: with `USE_DRIVE = True` finished folds and per-epoch state live on Drive and the run resumes.

Nothing here changes the repo: results come back as a zip and are committed from the Mac side.

In [ ]:
REPO_URL = "https://github.com/<you>/surgical_tool_id.git"   # <- fill in
BRANCH = "main"
STAGE = "3b"                                                   # <- stage to run
USE_DRIVE = True   # keep per-epoch run state on Google Drive so a disconnect can resume (one auth click)


In [ ]:
import os, subprocess, sys
!nvidia-smi -L || echo "no GPU - switch the runtime to T4 GPU"
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@")
except Exception:
    pass
!rm -rf /content/repo
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/content/repo"], check=True)
%cd /content/repo
!git log --oneline -1
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/surgical_tool_id_runs", exist_ok=True)
    !rm -rf /content/repo/runs && ln -s /content/drive/MyDrive/surgical_tool_id_runs /content/repo/runs
    print("run state on Drive: re-running this notebook after a disconnect resumes where it stopped")

In [ ]:
# ImageNet weights (training only); hashes must start with the tag in the filename
!mkdir -p weights
!curl -fsSL -o weights/resnet18-f37072fd.pth                  https://download.pytorch.org/models/resnet18-f37072fd.pth
!curl -fsSL -o weights/efficientnet_b0_rwightman-7f5810bc.pth https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth
!curl -fsSL -o weights/mobilenet_v3_large-5c1a4163.pth        https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth
import hashlib, glob
for p in sorted(glob.glob("weights/*.pth")):
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    tag = p.rsplit("-", 1)[1].split(".")[0]
    print(p, h[:8], "OK" if h.startswith(tag) else "HASH MISMATCH")
    assert h.startswith(tag)

In [ ]:
import sys; sys.path.insert(0, "/content/repo")
from toolid import experiments
exps = experiments.STAGES[STAGE]
print(f"stage {STAGE}: {len(exps)} experiments x 5 folds x 3 seeds")
for e in exps: print(" ", e)

In [ ]:
import time
t0 = time.time()
for e in exps:
    print(f"=== {e} ({(time.time()-t0)/60:.1f} min elapsed) ===", flush=True)
    !python scripts/cv.py --exp {e} 2>&1 | grep -v -i warn | tail -5
print(f"stage {STAGE} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# package results (json + OOF logits) and download
import zipfile, os
from google.colab import files
zp = f"/content/results_{STAGE}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for e in exps:
        for suf in (".json", "_oof.npz"):
            p = f"results/cv/{e}{suf}"
            if os.path.exists(p): z.write(p)
    z.writestr("COLAB_ENV.txt", subprocess.run([sys.executable, "-c",
        "import torch,torchvision;print(torch.__version__,torchvision.__version__,torch.cuda.get_device_name(0))"],
        capture_output=True, text=True).stdout + subprocess.run(["git","log","--oneline","-1"], capture_output=True, text=True).stdout)
print(zipfile.ZipFile(zp).namelist())
files.download(zp)